In [3]:
# ✅ Imports
import numpy as np, pandas as pd, torch, random, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Optional qiskit import (used if available)
try:
    from qiskit import QuantumCircuit, Aer, execute
    from qiskit.circuit import Parameter
    from qiskit.utils import QuantumInstance
    from qiskit.algorithms.optimizers import COBYLA
    QISKIT_AVAILABLE = True
except Exception:
    QISKIT_AVAILABLE = False

# ✅ Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load and Preprocess Data
def load_data(path):
    df = pd.read_csv(path).dropna(axis=1, how='all').dropna()
    df['Attack Type'] = df['Attack Type'].apply(lambda x: 0 if x == 'Normal Traffic' else 1)
    df = df.rename(columns={'Attack Type': 'label'})

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    X = StandardScaler().fit_transform(df.drop(columns=['label']))
    y = df['label'].values

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_idx, test_idx in sss.split(X, y):
        return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# ✅ BiLSTM + MultiHead Attention Model
# ✅ BiLSTM + MultiHead Attention Model (fixed: treat features as vector per time-step)
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, C)
        B, T, C = x.size()
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class IDSModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=2):
        """
        input_dim = number of features per sample (will be used as LSTM input_size)
        We treat each sample as a sequence of length 1 with feature-dimension=input_dim.
        """
        super().__init__()
        # LSTM input_size = number of features
        self.bilstm = nn.LSTM(input_size=input_dim,
                              hidden_size=hidden_dim,
                              num_layers=1,
                              batch_first=True,
                              bidirectional=True)
        # attention on top of LSTM outputs
        self.attn = MultiHeadSelfAttention(embed_dim=hidden_dim * 2, num_heads=4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # expect x shape: (batch, input_dim) OR (batch, seq_len, input_dim)
        if x.dim() == 2:
            x = x.unsqueeze(1)  # -> (batch, 1, input_dim)
        # LSTM expects (batch, seq_len, input_size)
        lstm_out, _ = self.bilstm(x)      # -> (batch, seq_len, hidden_dim*2)
        attn_out = self.attn(lstm_out)    # -> (batch, seq_len, hidden_dim*2)
        out = attn_out[:, -1, :]          # take last time step (seq_len=1 -> that step)
        return self.fc(out)



# === QAOA Feature Selection (replaces PSO) ===
# === QAOA Feature Selection (replaces PSO) with batched inference for memory safety ===
class QAOA:
    def __init__(self, model_fn, X, y, input_dim, num_samples=4, max_iter=4, qaoa_depth=1, infer_batch_size=4096):
        self.model_fn = model_fn
        self.X = X
        self.y = y
        self.input_dim = input_dim
        self.num_samples = num_samples
        self.max_iter = max_iter
        self.qaoa_depth = qaoa_depth
        self.cache = {}
        self.rng = np.random.RandomState(42)
        self.infer_batch_size = infer_batch_size  # batch size for inference to avoid OOM

    def evaluate(self, position):
        """Re-use your original PSO evaluate semantics exactly (including caching),
           but perform model inference in batches to avoid huge allocations."""
        key = tuple(position)
        if key in self.cache:
            return self.cache[key]

        idx = np.where(position == 1)[0]
        if len(idx) == 0:
            self.cache[key] = 0.0
            return 0.0

        X_sel = self.X[:, idx]   # shape (N_samples, n_selected)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.95, random_state=42)
        for sub_idx, _ in sss.split(X_sel, self.y):
            X_sample = X_sel[sub_idx]
            y_sample = self.y[sub_idx]

        model = self.model_fn(len(idx)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

        # quick train on one small batch (keeps original logic)
        loader = DataLoader(TensorDataset(torch.tensor(X_sample, dtype=torch.float32),
                                          torch.tensor(y_sample, dtype=torch.long)),
                            batch_size=256, shuffle=True)
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            break  # only 1 batch for speed

        # ---------------------------
        # batched inference on full X_sel (memory-safe)
        # ---------------------------
        model.eval()
        preds_list = []
        with torch.no_grad():
            N = X_sel.shape[0]
            bs = min(self.infer_batch_size, max(1, N))
            for start in range(0, N, bs):
                end = start + bs
                xb_np = X_sel[start:end]                          # keep as numpy on CPU
                xb = torch.tensor(xb_np, dtype=torch.float32).to(device)  # move only small batch to device
                out = model(xb)                                  # (batch, num_classes)
                p = torch.argmax(out, dim=1).cpu().numpy()
                preds_list.append(p)
        if len(preds_list) == 0:
            preds = np.array([], dtype=int)
        else:
            preds = np.concatenate(preds_list, axis=0)

        # compute score against full labels (self.y)
        score = f1_score(self.y, preds)
        self.cache[key] = score
        return score

    # classical fallback sampler (unchanged simple version)
    def _classical_qaoa_sample(self, angles):
        n = self.input_dim
        agg = sum(np.sin(a) for a in angles)
        base_prob = 0.5 + 0.5 * np.tanh(agg / max(1.0, self.qaoa_depth))
        probs = base_prob + 0.1 * np.sin(np.arange(n) + agg)
        probs = np.clip(probs, 0.01, 0.99)
        samples = []
        for _ in range(self.num_samples):
            b = (self.rng.rand(n) < probs).astype(int)
            samples.append(b)
        return np.array(samples)

    def _sample_bitstrings(self, angles):
        # we use classical fallback sampler for reliability in typical environments
        return self._classical_qaoa_sample(angles)

    def optimize(self):
        p = self.qaoa_depth
        num_params = 2 * p
        angles = self.rng.uniform(-np.pi, np.pi, size=num_params)
        best_position = np.zeros(self.input_dim, dtype=int)
        best_score = 0.0

        for it in range(self.max_iter):
            candidates = self._sample_bitstrings(angles)  # (num_samples, input_dim)
            for cand in candidates:
                score = self.evaluate(cand)
                if score > best_score:
                    best_score = score
                    best_position = cand.copy()
            # small perturbation / heuristic update
            angles += self.rng.normal(scale=0.05, size=num_params)

        return np.where(best_position == 1)[0]


# ✅ MAML-style Meta Training
def meta_train(model, X_train, y_train, epochs=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

    loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                      torch.tensor(y_train, dtype=torch.long)), batch_size=256, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model

# ✅ Final Evaluation
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device))
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    print("\n--- Final Evaluation ---")
    print(f"Accuracy : {accuracy_score(y_test, preds)*100:.2f}")
    print(f"Precision: {precision_score(y_test, preds)*100:.2f}")
    print(f"Recall   : {recall_score(y_test, preds)*100:.2f}")
    print(f"F1 Score : {f1_score(y_test, preds)*100:.2f}")

# ✅ Main Execution
if __name__ == "__main__":
    start = time.time()
    path = r"C:\Users\chari\Downloads\cicids2017_cleaned.csv"
    X_train, X_test, y_train, y_test = load_data(path)

    print("⚙️ Running QAOA-based feature selection...")
    # instantiate QAOA with same "spirit" as your previous PSO call
    qaoa = QAOA(IDSModel, X_train, y_train, input_dim=X_train.shape[1], num_samples=4, max_iter=4, qaoa_depth=1)
    selected_features = qaoa.optimize()
    print("✅ Selected features:", selected_features)

    X_train_sel = X_train[:, selected_features]
    X_test_sel = X_test[:, selected_features]

    print("🚀 Training MAML-based IDS Model...")
    model = IDSModel(X_train_sel.shape[1])
    model = meta_train(model, X_train_sel, y_train, epochs=8)

    evaluate_model(model, X_test_sel, y_test)
    print(f"\n⏱️ Total time: {(time.time() - start)/60:.2f} minutes")


⚙️ Running QAOA-based feature selection...
✅ Selected features: [ 2  6  8  9 10 12 18 23 28 31 32 33 34 35 36 38 40 44 50]
🚀 Training MAML-based IDS Model...

--- Final Evaluation ---
Accuracy : 98.79
Precision: 97.30
Recall   : 95.47
F1 Score : 96.37

⏱️ Total time: 13.29 minutes


In [1]:
# ✅ Imports
import numpy as np, pandas as pd, torch, random, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Optional qiskit import (used if available)
try:
    from qiskit import QuantumCircuit, Aer, execute
    from qiskit.circuit import Parameter
    from qiskit.utils import QuantumInstance
    from qiskit.algorithms.optimizers import COBYLA
    QISKIT_AVAILABLE = True
except Exception:
    QISKIT_AVAILABLE = False

# ✅ Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load and Preprocess Data
def load_data(path):
    df = pd.read_csv(path).dropna(axis=1, how='all').dropna()
    df['Attack Type'] = df['Attack Type'].apply(lambda x: 0 if x == 'Normal Traffic' else 1)
    df = df.rename(columns={'Attack Type': 'label'})

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    X = StandardScaler().fit_transform(df.drop(columns=['label']))
    y = df['label'].values

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_idx, test_idx in sss.split(X, y):
        return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# ✅ BiLSTM + MultiHead Attention Model
# ✅ BiLSTM + MultiHead Attention Model (fixed: treat features as vector per time-step)
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, C)
        B, T, C = x.size()
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class IDSModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=2):
        """
        input_dim = number of features per sample (will be used as LSTM input_size)
        We treat each sample as a sequence of length 1 with feature-dimension=input_dim.
        """
        super().__init__()
        # LSTM input_size = number of features
        self.bilstm = nn.LSTM(input_size=input_dim,
                              hidden_size=hidden_dim,
                              num_layers=1,
                              batch_first=True,
                              bidirectional=True)
        # attention on top of LSTM outputs
        self.attn = MultiHeadSelfAttention(embed_dim=hidden_dim * 2, num_heads=4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # expect x shape: (batch, input_dim) OR (batch, seq_len, input_dim)
        if x.dim() == 2:
            x = x.unsqueeze(1)  # -> (batch, 1, input_dim)
        # LSTM expects (batch, seq_len, input_size)
        lstm_out, _ = self.bilstm(x)      # -> (batch, seq_len, hidden_dim*2)
        attn_out = self.attn(lstm_out)    # -> (batch, seq_len, hidden_dim*2)
        out = attn_out[:, -1, :]          # take last time step (seq_len=1 -> that step)
        return self.fc(out)



# === QAOA Feature Selection (replaces PSO) ===
# === QAOA Feature Selection (replaces PSO) with batched inference for memory safety ===
class QAOA:
    def __init__(self, model_fn, X, y, input_dim, num_samples=4, max_iter=4, qaoa_depth=1, infer_batch_size=4096):
        self.model_fn = model_fn
        self.X = X
        self.y = y
        self.input_dim = input_dim
        self.num_samples = num_samples
        self.max_iter = max_iter
        self.qaoa_depth = qaoa_depth
        self.cache = {}
        self.rng = np.random.RandomState(42)
        self.infer_batch_size = infer_batch_size  # batch size for inference to avoid OOM

    def evaluate(self, position):
        """Re-use your original PSO evaluate semantics exactly (including caching),
           but perform model inference in batches to avoid huge allocations."""
        key = tuple(position)
        if key in self.cache:
            return self.cache[key]

        idx = np.where(position == 1)[0]
        if len(idx) == 0:
            self.cache[key] = 0.0
            return 0.0

        X_sel = self.X[:, idx]   # shape (N_samples, n_selected)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.95, random_state=42)
        for sub_idx, _ in sss.split(X_sel, self.y):
            X_sample = X_sel[sub_idx]
            y_sample = self.y[sub_idx]

        model = self.model_fn(len(idx)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

        # quick train on one small batch (keeps original logic)
        loader = DataLoader(TensorDataset(torch.tensor(X_sample, dtype=torch.float32),
                                          torch.tensor(y_sample, dtype=torch.long)),
                            batch_size=256, shuffle=True)
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            break  # only 1 batch for speed

        # ---------------------------
        # batched inference on full X_sel (memory-safe)
        # ---------------------------
        model.eval()
        preds_list = []
        with torch.no_grad():
            N = X_sel.shape[0]
            bs = min(self.infer_batch_size, max(1, N))
            for start in range(0, N, bs):
                end = start + bs
                xb_np = X_sel[start:end]                          # keep as numpy on CPU
                xb = torch.tensor(xb_np, dtype=torch.float32).to(device)  # move only small batch to device
                out = model(xb)                                  # (batch, num_classes)
                p = torch.argmax(out, dim=1).cpu().numpy()
                preds_list.append(p)
        if len(preds_list) == 0:
            preds = np.array([], dtype=int)
        else:
            preds = np.concatenate(preds_list, axis=0)

        # compute score against full labels (self.y)
        score = f1_score(self.y, preds)
        self.cache[key] = score
        return score

    # classical fallback sampler (unchanged simple version)
    def _classical_qaoa_sample(self, angles):
        n = self.input_dim
        agg = sum(np.sin(a) for a in angles)
        base_prob = 0.5 + 0.5 * np.tanh(agg / max(1.0, self.qaoa_depth))
        probs = base_prob + 0.1 * np.sin(np.arange(n) + agg)
        probs = np.clip(probs, 0.01, 0.99)
        samples = []
        for _ in range(self.num_samples):
            b = (self.rng.rand(n) < probs).astype(int)
            samples.append(b)
        return np.array(samples)

    def _sample_bitstrings(self, angles):
        # we use classical fallback sampler for reliability in typical environments
        return self._classical_qaoa_sample(angles)

    def optimize(self):
        p = self.qaoa_depth
        num_params = 2 * p
        angles = self.rng.uniform(-np.pi, np.pi, size=num_params)
        best_position = np.zeros(self.input_dim, dtype=int)
        best_score = 0.0

        for it in range(self.max_iter):
            candidates = self._sample_bitstrings(angles)  # (num_samples, input_dim)
            for cand in candidates:
                score = self.evaluate(cand)
                if score > best_score:
                    best_score = score
                    best_position = cand.copy()
            # small perturbation / heuristic update
            angles += self.rng.normal(scale=0.05, size=num_params)

        return np.where(best_position == 1)[0]


# ✅ MAML-style Meta Training
def meta_train(model, X_train, y_train, epochs=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

    loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                      torch.tensor(y_train, dtype=torch.long)), batch_size=256, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model

# ✅ Final Evaluation
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device))
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    print("\n--- Final Evaluation ---")
    print(f"Accuracy : {accuracy_score(y_test, preds)*100:.2f}")
    print(f"Precision: {precision_score(y_test, preds)*100:.2f}")
    print(f"Recall   : {recall_score(y_test, preds)*100:.2f}")
    print(f"F1 Score : {f1_score(y_test, preds)*100:.2f}")

# ✅ Main Execution
if __name__ == "__main__":
    start = time.time()
    path = r"C:\Users\chari\Downloads\cicids2017_cleaned.csv"
    X_train, X_test, y_train, y_test = load_data(path)
    model = meta_train(model, X_train_sel, y_train, epochs=8)

    evaluate_model(model, X_test_sel, y_test)

    print("⚙️ Running QAOA-based feature selection...")
    # instantiate QAOA with same "spirit" as your previous PSO call
    qaoa = QAOA(IDSModel, X_train, y_train, input_dim=X_train.shape[1], num_samples=4, max_iter=4, qaoa_depth=1)
    selected_features = qaoa.optimize()
    print("✅ Selected features:", selected_features)

    X_train_sel = X_train[:, selected_features]
    X_test_sel = X_test[:, selected_features]

    print("🚀 Training MAML-based IDS Model...")
    model = IDSModel(X_train_sel.shape[1])
    print(f"\n⏱️ Total time: {(time.time() - start)/60:.2f} minutes")


⚙️ Running QAOA-based feature selection...
✅ Selected features: [ 2  6  8  9 10 12 18 23 28 31 32 33 34 35 36 38 40 44 50]
🚀 Training MAML-based IDS Model...

--- Final Evaluation ---
Accuracy : 98.79
Precision: 97.30
Recall   : 95.47
F1 Score : 96.37

⏱️ Total time: 15.60 minutes


In [2]:
# 📦 Cell 1: Imports and Setup
# - Import necessary libraries (PyTorch, sklearn, qiskit if available, etc.)
# - Set reproducibility seed
# - Choose device (GPU if available)

import numpy as np, pandas as pd, torch, random, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Optional qiskit import (used if available)
try:
    from qiskit import QuantumCircuit, Aer, execute
    from qiskit.circuit import Parameter
    from qiskit.utils import QuantumInstance
    from qiskit.algorithms.optimizers import COBYLA
    QISKIT_AVAILABLE = True
except Exception:
    QISKIT_AVAILABLE = False

# ✅ Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
# 📦 Cell 2: Data Loading and Preprocessing
# - Load CSV dataset
# - Encode categorical features
# - Standardize numerical features
# - Stratified train-test split

def load_data(path):
    df = pd.read_csv(path).dropna(axis=1, how='all').dropna()
    df['Attack Type'] = df['Attack Type'].apply(lambda x: 0 if x == 'Normal Traffic' else 1)
    df = df.rename(columns={'Attack Type': 'label'})

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    X = StandardScaler().fit_transform(df.drop(columns=['label']))
    y = df['label'].values

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_idx, test_idx in sss.split(X, y):
        return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


In [4]:
# 📦 Cell 3: Multi-Head Self-Attention Layer
# - Defines custom self-attention block to enhance BiLSTM outputs

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, C)
        B, T, C = x.size()
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out)


In [5]:
# 📦 Cell 4: IDS Model (BiLSTM + Multi-Head Attention)
# - BiLSTM extracts temporal dependencies
# - Multi-head attention improves representation
# - Final FC layer for classification

class IDSModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=2):
        super().__init__()
        self.bilstm = nn.LSTM(input_size=input_dim,
                              hidden_size=hidden_dim,
                              num_layers=1,
                              batch_first=True,
                              bidirectional=True)
        self.attn = MultiHeadSelfAttention(embed_dim=hidden_dim * 2, num_heads=4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)  
        lstm_out, _ = self.bilstm(x)
        attn_out = self.attn(lstm_out)
        out = attn_out[:, -1, :]  
        return self.fc(out)


In [6]:
# 📦 Cell 5: QAOA Feature Selection (classical approximation)
# - Selects best feature subset using QAOA-inspired classical heuristic
# - Includes batched inference to avoid OOM
# - Evaluates candidate subsets via F1-score

class QAOA:
    def __init__(self, model_fn, X, y, input_dim, num_samples=4, max_iter=4, qaoa_depth=1, infer_batch_size=4096):
        self.model_fn = model_fn
        self.X = X
        self.y = y
        self.input_dim = input_dim
        self.num_samples = num_samples
        self.max_iter = max_iter
        self.qaoa_depth = qaoa_depth
        self.cache = {}
        self.rng = np.random.RandomState(42)
        self.infer_batch_size = infer_batch_size  

    def evaluate(self, position):
        key = tuple(position)
        if key in self.cache:
            return self.cache[key]

        idx = np.where(position == 1)[0]
        if len(idx) == 0:
            self.cache[key] = 0.0
            return 0.0

        X_sel = self.X[:, idx]   
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.95, random_state=42)
        for sub_idx, _ in sss.split(X_sel, self.y):
            X_sample = X_sel[sub_idx]
            y_sample = self.y[sub_idx]

        model = self.model_fn(len(idx)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

        loader = DataLoader(TensorDataset(torch.tensor(X_sample, dtype=torch.float32),
                                          torch.tensor(y_sample, dtype=torch.long)),
                            batch_size=256, shuffle=True)
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            break  

        model.eval()
        preds_list = []
        with torch.no_grad():
            N = X_sel.shape[0]
            bs = min(self.infer_batch_size, max(1, N))
            for start in range(0, N, bs):
                end = start + bs
                xb = torch.tensor(X_sel[start:end], dtype=torch.float32).to(device)
                out = model(xb)
                p = torch.argmax(out, dim=1).cpu().numpy()
                preds_list.append(p)
        preds = np.concatenate(preds_list, axis=0)

        score = f1_score(self.y, preds)
        self.cache[key] = score
        return score

    def _classical_qaoa_sample(self, angles):
        n = self.input_dim
        agg = sum(np.sin(a) for a in angles)
        base_prob = 0.5 + 0.5 * np.tanh(agg / max(1.0, self.qaoa_depth))
        probs = base_prob + 0.1 * np.sin(np.arange(n) + agg)
        probs = np.clip(probs, 0.01, 0.99)
        return np.array([(self.rng.rand(n) < probs).astype(int) for _ in range(self.num_samples)])

    def _sample_bitstrings(self, angles):
        return self._classical_qaoa_sample(angles)

    def optimize(self):
        p = self.qaoa_depth
        num_params = 2 * p
        angles = self.rng.uniform(-np.pi, np.pi, size=num_params)
        best_position = np.zeros(self.input_dim, dtype=int)
        best_score = 0.0

        for _ in range(self.max_iter):
            candidates = self._sample_bitstrings(angles)
            for cand in candidates:
                score = self.evaluate(cand)
                if score > best_score:
                    best_score = score
                    best_position = cand.copy()
            angles += self.rng.normal(scale=0.05, size=num_params)

        return np.where(best_position == 1)[0]


In [7]:
# 📦 Cell 6: Meta-Training Function
# - Simple MAML-style meta training for IDS model

def meta_train(model, X_train, y_train, epochs=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.4, 0.6]).to(device))

    loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                      torch.tensor(y_train, dtype=torch.long)), batch_size=256, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


In [8]:
# 📦 Cell 7: Final Evaluation Function
# - Evaluate trained IDS model on test set
# - Print accuracy, precision, recall, F1

def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32).to(device))
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    print("\n--- Final Evaluation ---")
    print(f"Accuracy : {accuracy_score(y_test, preds)*100:.2f}")
    print(f"Precision: {precision_score(y_test, preds)*100:.2f}")
    print(f"Recall   : {recall_score(y_test, preds)*100:.2f}")
    print(f"F1 Score : {f1_score(y_test, preds)*100:.2f}")


In [9]:
# 📦 Cell 8: Main Execution
# - Load data
# - Run QAOA-based feature selection
# - Train IDS model with selected features
# - Evaluate on test data

if __name__ == "__main__":
    start = time.time()
    path = r"C:\Users\chari\Downloads\cicids2017_cleaned.csv"
    X_train, X_test, y_train, y_test = load_data(path)

    print("⚙️ Running QAOA-based feature selection...")
    qaoa = QAOA(IDSModel, X_train, y_train, input_dim=X_train.shape[1], num_samples=4, max_iter=4, qaoa_depth=1)
    selected_features = qaoa.optimize()
    print("✅ Selected features:", selected_features)

    X_train_sel = X_train[:, selected_features]
    X_test_sel = X_test[:, selected_features]

    print("🚀 Training MAML-based IDS Model...")
    model = IDSModel(X_train_sel.shape[1])
    model = meta_train(model, X_train_sel, y_train, epochs=8)

    evaluate_model(model, X_test_sel, y_test)
    print(f"\n⏱️ Total time: {(time.time() - start)/60:.2f} minutes")


⚙️ Running QAOA-based feature selection...
✅ Selected features: [ 2  6  8  9 10 12 18 23 28 31 32 33 34 35 36 38 40 44 50]
🚀 Training MAML-based IDS Model...

--- Final Evaluation ---
Accuracy : 98.79
Precision: 97.30
Recall   : 95.47
F1 Score : 96.37

⏱️ Total time: 20.91 minutes
